In [13]:
from collections import defaultdict
from pathlib import Path
import csv
import openpyxl

In [14]:
SRC_PATH = Path("new_dataset_brend_normalized_finalv13.xlsx")
DST_XLSX = Path("dataset_final.xlsx")
DST_CSV = Path("dataset_final.csv")  

ALL_YEARS = list(range(2010, 2026))
MONTHS_SOURCE = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTHS_OUTPUT = [m.lower() for m in MONTHS_SOURCE]
SKIP_BRANDS = {"TOTAL", "CUMULATIVE", "Brand"}

In [15]:
def clean_text(value):
    if value is None:
        return None
    text = str(value).replace("\xa0", " ").strip()
    return text if text else None


def clean_fuel(value):
    text = clean_text(value)
    return text.lower() if text is not None else None


def to_int(value):
    if value is None:
        return 0
    try:
        return int(float(str(value).replace(",", ".")))
    except (TypeError, ValueError):
        return 0


def clean_cc(value):
    if value is None or str(value).strip() == "":
        return None
    try:
        return int(float(str(value).replace(",", ".")))
    except (TypeError, ValueError):
        return clean_text(value)

In [16]:
def parse_source(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    print(f"Reading source: {path}")
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)

    
    data_map = defaultdict(lambda: defaultdict(lambda: [0] * 12))
    ordered_keys = []
    seen_keys = set()

    for sheet_name in wb.sheetnames:
        try:
            year = int(str(sheet_name).strip())
        except ValueError:
            continue

        if year not in ALL_YEARS:
            continue

        ws = wb[sheet_name]
        current_brand = None

        for row in ws.iter_rows(min_row=2, values_only=True):
            brand_raw = row[0] if len(row) > 0 else None
            model_raw = row[1] if len(row) > 1 else None
            fuel_raw = row[2] if len(row) > 2 else None
            cc_raw = row[16] if len(row) > 16 else None  # Q column = cc

            if brand_raw is None and model_raw is None:
                continue

            brand_text = clean_text(brand_raw)
            if brand_text in SKIP_BRANDS:
                continue

            
            if brand_text is not None:
                current_brand = brand_text

            model = clean_text(model_raw)
            if current_brand is None or model is None:
                continue

            fuel = clean_fuel(fuel_raw)
            cc = clean_cc(cc_raw)
            key = (current_brand, model, fuel, cc)

            if key not in seen_keys:
                seen_keys.add(key)
                ordered_keys.append(key)

            
            for month_idx in range(12):
                source_col = 3 + month_idx
                value = row[source_col] if len(row) > source_col else None
                data_map[key][year][month_idx] += to_int(value)

    wb.close()
    print(f"Unique brand-model-fuel-cc keys: {len(ordered_keys):,}")
    return ordered_keys, data_map

In [17]:
def write_xlsx(ordered_keys, data_map, output_path):
    print(f"Writing XLSX: {output_path}")

    wb_out = openpyxl.Workbook(write_only=True)
    ws_out = wb_out.create_sheet("Dataset")

    ws_out.append(["year", "month", "sum", "brand", "model", "fuel", "cc"])

    rows_written = 0
    for brand, model, fuel, cc in ordered_keys:
        key = (brand, model, fuel, cc)
        for year in ALL_YEARS:
            monthly_values = data_map[key].get(year, [0] * 12)
            for month_name, sales in zip(MONTHS_OUTPUT, monthly_values):
                ws_out.append([year, month_name, sales, brand, model, fuel, cc])
                rows_written += 1

    wb_out.save(output_path)
    print(f"Data rows written: {rows_written:,}")
    return rows_written


def write_csv(ordered_keys, data_map, output_path):
    if output_path is None:
        return None

    print(f"Writing CSV: {output_path}")
    rows_written = 0

    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["year", "month", "sum", "brand", "model", "fuel", "cc"])

        for brand, model, fuel, cc in ordered_keys:
            key = (brand, model, fuel, cc)
            for year in ALL_YEARS:
                monthly_values = data_map[key].get(year, [0] * 12)
                for month_name, sales in zip(MONTHS_OUTPUT, monthly_values):
                    writer.writerow([year, month_name, sales, brand, model, fuel, cc])
                    rows_written += 1

    print(f"CSV rows written: {rows_written:,}")
    return rows_written

In [18]:
ordered_keys, data_map = parse_source(SRC_PATH)

rows_xlsx = write_xlsx(ordered_keys, data_map, DST_XLSX)
rows_csv = write_csv(ordered_keys, data_map, DST_CSV)

print("\nDone")
print(f"Rows in dataset: {rows_xlsx:,}")
print(f"Columns: 7")
print(f"XLSX: {DST_XLSX}")
if DST_CSV is not None:
    print(f"CSV: {DST_CSV}")

Reading source: new_dataset_brend_normalized_finalv13.xlsx
Unique brand-model-fuel-cc keys: 5,245
Writing XLSX: dataset_final.xlsx
Data rows written: 1,007,040
Writing CSV: dataset_final.csv
CSV rows written: 1,007,040

Done
Rows in dataset: 1,007,040
Columns: 7
XLSX: dataset_final.xlsx
CSV: dataset_final.csv


In [19]:
check_wb = openpyxl.load_workbook(DST_XLSX, read_only=True, data_only=True)
ws = check_wb["Dataset"]

print("Rows including header:", ws.max_row)
print("Columns:", ws.max_column)

for row in ws.iter_rows(min_row=1, max_row=10, values_only=True):
    print(row)

check_wb.close()

Rows including header: None
Columns: None
('year', 'month', 'sum', 'brand', 'model', 'fuel', 'cc')
(2010, 'jan', 25, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'feb', 50, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'mar', 250, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'apr', 260, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'may', 125, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'jun', 67, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'jul', 84, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'aug', 82, 'CHEVROLET', 'Kalos LS', 'g', 1400)
(2010, 'sep', 100, 'CHEVROLET', 'Kalos LS', 'g', 1400)
